<a href="https://colab.research.google.com/github/hfmohammed18/ArcticInference/blob/main/Generate_Music.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### © Copyright 2021-present Aditya Gomatam

This file is part of music-transformer (https://github.com/spectraldoy/music-transformer), my project to build and
train a Music Transformer. music-transformer is open-source software licensed under the terms of the GNU General
Public License v3.0. music-transformer is free software: you can redistribute it and/or modify it under the terms of
the GNU General Public License as published by the Free Software Foundation, either version 3 of the License,
or (at your option) any later version. music-transformer is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
See the GNU General Public License for more details. A copy of this license can be found within the GitHub repository
for music-transformer, or at https://www.gnu.org/licenses/gpl-3.0.html.

# Generate Music with Music Transformer!

If you're seeing this on the GitHub repository:

1. Click the Open in Colab link
2. The notebook should be pre-configured with a GPU, but in case it isn't, go to Runtime > Change runtime type, and select GPU for Hardware accelerator
3. Run the cells you wish to run with the Play buttons at their top lefts

This Notebook lets you play with pretrained [Music Transformer](https://arxiv.org/pdf/1809.04281.pdf) models to generate piano music.

There are a few models available on the [GitHub repository](https://github.com/spectraldoy/music-transformer/tree/main/models), of which the one trained on Video Game Music is automatically loaded, and with which you can immediately generate music. If you wish, you may edit the code under the markdown cells in this notebook to generate music with another of those models, or even upload your own to generate music with. I find that all of my models are highly prone to repeated notes and wandering melodies, but their harmonies are consistent.

In [1]:
#@title Set up Generation
#@markdown Clone into the GitHub repository, import required libraries,
#@markdown download the Google Magenta SoundFont, and set up variables
#@markdown for music generation. You must run this cell if you want to generate any music..
%%capture

import os
from google.colab import files
from random import randint
from IPython.display import Audio

!gsutil -q -m cp gs://magentadata/soundfonts/Yamaha-C5-Salamander-JNv5.1.sf2 ./

!apt install fluidsynth

!git clone https://github.com/spectraldoy/music-transformer

!pip install mido

gen_path = "./gen_audio.mid"
wav_path = "./gen_audio.wav"
model_path = "./music-transformer/models/vgmtransformerv4.pt"

In [2]:
!unzip /content/mega_tensor.zip

Archive:  /content/mega_tensor.zip
  inflating: mega_tensor.pt          


In [6]:
!python ./music-transformer/preprocessing.py data ./music-transformer/tensor.pt 512  -v

# self.datapath = datapath
# self.batch_size = batch_size
# # print("checkkkk --------",torch.load(datapath))
# tensor_list=torch.load(datapath)
# padded = [
#     F.pad(t, (0, 512 - t.size(0)), value=0)
#     for t in tensor_list
# ]
# full_batch = torch.stack(padded, dim=0)
# #tempdata=torch.stack(tensor_list, dim=0)
# data = full_batch.long().to(device)


Translating midi files to event vocabulary (NOTE: may take a while)...
data/221_MarioBros__05_06GameOver.mid
data/221_MarioBros__01_02GameStartA.mid
data/221_MarioBros__04_05Restart.mid
data/221_MarioBros__02_03GameStartB.mid
data/221_MarioBros__03_04Perfect.mid
Done!
Randomly sampling and cutting data to length...
Done!
Augmenting data (NOTE: may take even longer)...
Done!
Saving...
Done!


In [7]:
!python ./music-transformer/train.py /content/mega_tensor.pt /content/music-transformer/models/model4v2.pt output.pt 10

Setting up the trainer...
There are 9004 samples in the data, 7203 training samples and 1801 validation samples

Beginning training...
2025-05-30 07:52
W0530 07:52:24.727000 4263 torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode
Epoch 0 Time taken 168.73 seconds Train Loss 4.051261537897903 Val Loss 1.9456920017275894
Epoch 1 Time taken 32.83 seconds Train Loss 1.6683535649713161 Val Loss 1.4934316894464326
Epoch 2 Time taken 33.54 seconds Train Loss 1.4317771802961299 Val Loss 1.369366074863233
Epoch 3 Time taken 32.39 seconds Train Loss 1.314271255932023 Val Loss 1.2475134667597318
Epoch 4 Time taken 32.92 seconds Train Loss 1.1603030364597793 Val Loss 1.059318863508994
Epoch 5 Time taken 32.88 seconds Train Loss 0.9862716838849329 Val Loss 0.919784380678545
Epoch 6 Time taken 32.58 seconds Train Loss 0.8855741548854693 Val Loss 0.8435937992313451
Epoch 7 Time taken 32.64 seconds Train Loss 0.8141802406944005 Val Loss 0.7700605277429547
Epoch 8 Time ta

In [8]:
!python ./music-transformer/generate.py /content/output.pt output.mid

Done


In [9]:
print("Creating playable audio...")
os.system(f"fluidsynth -ni Yamaha-C5-Salamander-JNv5.1.sf2 output.mid -F {wav_path} -r 44100 -g 1.0")
Audio(wav_path)

Creating playable audio...


In [2]:
#@title Generate a Piano Performance from Scratch!

#@markdown The Music Transformer
#@markdown is an autoregressive model, which means it generates outputs one
#@markdown at a time, then looks to all its previous outputs in order to
#@markdown generate the rest. This process should take about a minute or so on a GPU in the best case.
#@markdown Sometimes these models just don't stop generating, which is why you can also KeyboardInterrupt,
#@markdown or press the stop button at the top left to interrupt execution and save whatever has already been
#@markdown generated.
#@markdown
#@markdown Additionally, you can set the approximate tempo here, but the actual tempo
#@markdown of the output will depend greatly on what notes and what rhythm
#@markdown are generated by the model.

tempo = 120 #@param {type:'slider', min:32, max:400}

#@markdown Note that
#@markdown the model cannot generate complete pieces, but only
#@markdown abrupt "sections" of pieces. This is because only
#@markdown about 2000 MIDI events could be input at a time
#@markdown during training, whereas most pieces consist of 10 000 - 100 000 MIDI events.

temp = randint(1000, 1100) / 1000
!python ./music-transformer/generate.py {model_path} {gen_path} -v -t {temp} -tm {tempo}

print("Creating playable audio...")
os.system(f"fluidsynth -ni Yamaha-C5-Salamander-JNv5.1.sf2 {gen_path} -F {wav_path} -r 44100 -g 1.0")
Audio(wav_path)

Greedy decoding...
Generated 921 tokens. Time taken: 6.23 secs.
Saving midi file at ./gen_audio.mid...
Done
Creating playable audio...


In [ ]:
#@title Download Performance as MIDI file
#@markdown You can download the generated .wav file by
#@markdown clicking on the 3 dots in the displayed
#@markdown Audio. Run this cell to download
#@markdown the performance as MIDI.
files.download(gen_path)